# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their Croissant `@id` for clarity and reproducibility.

### Dataset Source
The dataset schema is provided as a Croissant JSON-LD file, accessible via the following URL:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset's metadata and access its contents using `mlcroissant`, referencing each data entity by its unique Croissant `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset METADATA
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Dataset summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the record sets, available fields, and their unique `@id`s in the Croissant schema. 

This helps to plan extraction and analysis steps.


In [ ]:
# Retrieve all record sets and their @id
from collections.abc import Iterable
def flatten(x):
    if isinstance(x, dict):
        for v in x.values():
            yield from flatten(v)
    elif isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
        for v in x:
            yield from flatten(v)
    else:
        yield x

record_sets = [r for r in dataset.record_sets]
print(f"Total number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"Record set @id: {rs.id}")
    print(f"Fields (with @id):")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("--"*20)


## 3. Data Extraction
Load the data from a specific record set (using its Croissant `@id`) to a DataFrame. Reference all fields by their `@id` for clarity. 

**Below, we extract data from each record set into separate DataFrames.**

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = dict()
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded data for record set @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")
    print(f"Fields: {list(dataframes[record_set_id].columns)}\n")

# Pick the main table (first record set, typically the core table in Croissant, adjust if needed):
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main Record Set ID: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets found!")

if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some standard EDA steps to the selected record set. We'll reference field and record set names by their `@id`s for clarity and reproducibility.

We'll select a numeric field (for example, age at second primary CRC as `dv:ageAt2ndCRC`, if present), filter records, normalize them, and group by another field such as anatomical site (e.g., `dv:anatomicalLocation2ndCRC`).


In [ ]:
# --- Specify field @ids here for clarity ---
# (adjust these as per the dataset overview printed in previous step; update accordingly if @ids differ)

numeric_field_id = None
group_field_id = None
fields_list = list(dataframes[main_record_set_id].columns) if main_record_set_id else []

# Try to auto-detect a numeric field (@id typically contains 'age' or similar keywords)
for col in fields_list:
    if 'age' in col.lower() or 'Age' in col:
        numeric_field_id = col
        break
# Try to auto-detect a grouping field (e.g., 'anatomical' or 'Site')
for col in fields_list:
    if ('anatomical' in col.lower()) or ('site' in col.lower()):
        group_field_id = col
        break

# For demonstration, print selection:
print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Grouping field selected (by @id): {group_field_id}")

if numeric_field_id and main_record_set_id:
    try:
        # Convert to numeric, coerce errors
        df = dataframes[main_record_set_id].copy()
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.25) # use Q1 as a threshold for demo
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field in the filtered data
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # If suitable, group by the detected group field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped (mean) data by {group_field_id}:\n")
            display(grouped_df[[numeric_field_id, norm_col]])
    except Exception as e:
        print(f"Could not perform EDA: {e}")
else:
    print("Could not determine a numeric field or main record set for EDA.")

## 5. Visualization
Create simple visualizations to display the distribution of the selected numeric field and its relationship with the chosen grouping field, referencing all axes and fields by their Croissant `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (numeric_field_id and main_record_set_id) and (numeric_field_id in dataframes[main_record_set_id].columns):
    df = dataframes[main_record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have demonstrated loading metadata, exploring record sets and fields via their Croissant `@id`, extracting tabular data, running exploratory data analysis, and visualizing core variables of the FAIR^2 dataset using the `mlcroissant` library.

**Key Takeaways:**
- All dataset components were referenced using their unique Croissant `@id` fields, ensuring clarity and reproducibility.
- This notebook provides a template for investigating Croissant datasets in a standards-based, programmatic workflow.

Feel free to extend this notebook by exploring further record sets, fields, and analyses as required for your study or machine learning workflow.
